# Chapter 09: Continuous Batching & PagedAttention

**Goal:** Understand how to efficiently serve LLMs to many concurrent users — moving from single-request inference to production-scale serving.

Single-request inference wastes the GPU. From Chapter 1, we know that at batch=1 during decode, the GPU is ~1% utilized (memory-bound, far below the ridge point). To serve many users, we need batching. But LLM requests have **variable lengths** — how do you batch efficiently?

This chapter covers the two key innovations that make modern LLM serving work: **continuous batching** and **PagedAttention**.

**Approach:** Fill in every `# YOUR CODE` section yourself — that's the learning.

**Model:** LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers)

**Hardware:** A100 80GB SXM4
- Measured HBM bandwidth: 1774 GB/s
- FP16 TFLOPS: 237
- Ridge point: ~134 FLOPS/byte

## 1. The Serving Problem

At batch=1 decode, the GPU is ~1% utilized — we load **all** model weights just to produce one token.
The obvious fix: batch multiple requests together so the same weight load serves many users.

But LLM requests are fundamentally different from image classification:
- Prompts have **variable lengths** (10 to 10,000 tokens)
- Generation lengths are **unknown** in advance
- Requests **arrive at different times**
- Each request has its own **KV cache** that grows during generation

How do you batch efficiently when nothing is the same shape?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model params (LLaMA-3.2-1B)
d_model = 2048
n_heads = 32
n_kv_heads = 8
d_head = 64
d_ff = 8192
n_layers = 16

# A100 specs
hbm_bandwidth = 1774  # GB/s
peak_tflops = 237     # FP16 TFLOPS
ridge_point = peak_tflops * 1000 / hbm_bandwidth  # ~134 FLOPS/byte

# Model weight size (FP16)
# Per layer: 7 linear projections
params_per_layer = (d_model * d_model * 2  # Q + O
                    + d_model * (n_kv_heads * d_head) * 2  # K + V
                    + d_model * d_ff * 3)  # gate + up + down
total_params = params_per_layer * n_layers
weight_bytes = total_params * 2  # FP16

print(f"Model weights: {weight_bytes / 1e9:.2f} GB")
print(f"Time to load all weights once: {weight_bytes / (hbm_bandwidth * 1e9) * 1000:.2f} ms")
print(f"Ridge point: {ridge_point:.0f} FLOPS/byte")
print(f"\nAt batch=1 decode: intensity ~1 FLOP/byte -> ~{1/ridge_point*100:.1f}% of peak compute")
print(f"At batch=128 decode: intensity ~128 FLOP/byte -> ~{min(128/ridge_point*100, 100):.0f}% of peak compute")

## 2. Static vs Continuous Batching

**Static batching:** Collect N requests, pad all to same length, run as a batch. Wait for ALL to finish before starting next batch.

Problems:
- Short requests wait for the longest one (tail latency)
- Padding wastes compute on useless tokens
- New requests must wait for current batch to fully complete

**Continuous batching (a.k.a. iteration-level scheduling):** Each decode step, check if any request finished. If so, immediately slot a new request into its place. No padding, no waiting.

Simulate both approaches to see the difference.

In [ ]:
np.random.seed(42)

# Simulate 10 requests with varying generation lengths
n_requests = 10
max_batch = 4  # max concurrent requests

# --- YOUR CODE: generate random request lengths ---
# Hint: gen_lengths = np.random.randint(10, 200, size=n_requests)
gen_lengths = None  # YOUR CODE

print(f"Request generation lengths: {gen_lengths}")
print(f"Mean: {gen_lengths.mean():.0f}, Max: {gen_lengths.max()}, Min: {gen_lengths.min()}")

# ---- STATIC BATCHING ----
# Process in fixed batches of max_batch, pad each batch to longest request
static_total_steps = 0
static_wasted_steps = 0

# --- YOUR CODE: simulate static batching ---
# Hint: chunk requests into groups of max_batch
# For each batch, total steps = batch_size * max_length_in_batch
# Wasted steps = total steps - sum of actual lengths in batch
for i in range(0, n_requests, max_batch):
    batch = gen_lengths[i:i+max_batch]
    batch_max = None       # YOUR CODE: batch.max()
    batch_total = None     # YOUR CODE: len(batch) * batch_max
    batch_useful = None    # YOUR CODE: batch.sum()
    static_total_steps += batch_total
    static_wasted_steps += (batch_total - batch_useful)

static_utilization = (static_total_steps - static_wasted_steps) / static_total_steps * 100

# ---- CONTINUOUS BATCHING ----
# Requests join/leave the batch dynamically each step
remaining = list(gen_lengths.copy())
queue = list(range(n_requests))
active = []  # indices into remaining
cont_total_steps = 0
cont_useful_steps = 0

# --- YOUR CODE: simulate continuous batching ---
# Hint: each iteration, fill active slots from queue, decrement remaining,
# remove finished requests, count useful steps = len(active) each iteration
while queue or active:
    # Fill empty slots
    while len(active) < max_batch and queue:
        active.append(queue.pop(0))

    # One decode step
    cont_total_steps += max_batch  # GPU capacity
    cont_useful_steps += len(active)  # YOUR CODE: len(active)

    # Decrement and remove finished
    finished = []
    for idx in active:
        remaining[idx] -= 1
        if remaining[idx] <= 0:
            finished.append(idx)
    for idx in finished:
        active.remove(idx)

cont_utilization = cont_useful_steps / cont_total_steps * 100

print(f"\n{'Static Batching':=^50}")
print(f"Total compute steps: {static_total_steps}")
print(f"Wasted steps (padding): {static_wasted_steps}")
print(f"GPU utilization: {static_utilization:.1f}%")

print(f"\n{'Continuous Batching':=^50}")
print(f"Total compute steps: {cont_total_steps}")
print(f"Useful steps: {cont_useful_steps}")
print(f"GPU utilization: {cont_utilization:.1f}%")
print(f"\nContinuous batching improvement: {cont_utilization / static_utilization:.2f}x")

## 3. The KV Cache Memory Problem

Each request maintains its own KV cache that **grows** during generation.

**KV cache size per token per layer:**
- K: n_kv_heads x d_head = 8 x 64 = 512 values
- V: same = 512 values
- Total: 1024 values x 2 bytes (FP16) = 2048 bytes per layer
- All 16 layers: 2048 x 16 = 32,768 bytes = 32 KB per token

With **static allocation**, you must pre-allocate space for `max_seq_len` tokens for every request — even though most requests use far fewer tokens. This is massively wasteful.

In [ ]:
# KV cache size calculations for LLaMA-3.2-1B

# --- YOUR CODE: calculate KV cache size per token ---
# Hint: kv_per_token_per_layer = 2 * n_kv_heads * d_head * 2  (K+V, FP16)
kv_per_token_per_layer = None  # YOUR CODE
kv_per_token = None            # YOUR CODE: kv_per_token_per_layer * n_layers

print(f"KV cache per token per layer: {kv_per_token_per_layer} bytes")
print(f"KV cache per token (all layers): {kv_per_token / 1024:.1f} KB")

# Simulate 10 concurrent requests
n_concurrent = 10
max_seq_len = 4096

# Actual lengths are much shorter on average
np.random.seed(42)
actual_lengths = np.random.exponential(scale=512, size=n_concurrent).astype(int)
actual_lengths = np.clip(actual_lengths, 50, max_seq_len)

# --- YOUR CODE: calculate wasted memory ---
# Static: pre-allocate max_seq_len for ALL requests
static_mem = None   # YOUR CODE: n_concurrent * max_seq_len * kv_per_token
# Dynamic: only allocate what's actually used
actual_mem = None    # YOUR CODE: sum(actual_lengths) * kv_per_token
wasted_mem = None    # YOUR CODE: static_mem - actual_mem
waste_pct = None     # YOUR CODE: wasted_mem / static_mem * 100

print(f"\n{'KV Cache Memory Analysis':=^55}")
print(f"Max sequence length: {max_seq_len}")
print(f"Actual lengths: {actual_lengths}")
print(f"Average actual length: {actual_lengths.mean():.0f} tokens")
print(f"\nStatic allocation:  {static_mem / 1e9:.2f} GB")
print(f"Actual needed:      {actual_mem / 1e9:.2f} GB")
print(f"Wasted memory:      {wasted_mem / 1e9:.2f} GB ({waste_pct:.1f}%)")
print(f"\nOn an 80 GB A100:")
print(f"  Static: could fit {int(80e9 / (max_seq_len * kv_per_token))} concurrent requests")
print(f"  Dynamic: could fit {int(80e9 / (actual_lengths.mean() * kv_per_token))} concurrent requests")

## 4. PagedAttention: Virtual Memory for KV Cache

Inspired by **OS virtual memory**: instead of allocating one big contiguous block per request,
allocate KV cache in fixed-size **blocks (pages)**, and use a **page table** to map logical
token positions to physical memory blocks.

Key ideas:
- **Block size**: e.g., 16 tokens per block
- **Page table**: maps (request, logical_block_idx) -> physical_block_idx
- **On-demand allocation**: only allocate a new physical block when the current one fills up
- **No contiguous memory needed**: blocks can be anywhere in GPU memory
- **No internal fragmentation**: only the last block of each request may be partially filled

This is exactly how virtual memory works in your OS — pages of physical RAM mapped by page tables.

In [ ]:
class SimplePagedKVCache:
    """Simplified PagedAttention page table."""

    def __init__(self, block_size, total_blocks):
        self.block_size = block_size
        self.total_blocks = total_blocks
        # --- YOUR CODE: initialize data structures ---
        # Hint: free_blocks = list(range(total_blocks))
        # Hint: page_tables = {}  # request_id -> [physical_block_ids]
        # Hint: seq_lengths = {}  # request_id -> current sequence length
        self.free_blocks = None  # YOUR CODE
        self.page_tables = None  # YOUR CODE
        self.seq_lengths = None  # YOUR CODE

    def allocate_request(self, request_id, initial_tokens=0):
        """Start tracking a new request."""
        self.page_tables[request_id] = []
        self.seq_lengths[request_id] = 0
        # Allocate blocks for initial tokens
        for _ in range(initial_tokens):
            self.append_token(request_id)

    def append_token(self, request_id):
        """Add one token to a request's KV cache."""
        seq_len = self.seq_lengths[request_id]
        # --- YOUR CODE: check if we need a new block ---
        # Hint: if seq_len % self.block_size == 0, allocate new block
        if seq_len % self.block_size == 0:
            if not self.free_blocks:
                raise RuntimeError("Out of physical blocks!")
            new_block = None  # YOUR CODE: self.free_blocks.pop(0)
            self.page_tables[request_id].append(new_block)
        self.seq_lengths[request_id] += 1

    def free_request(self, request_id):
        """Free all blocks for a completed request."""
        # --- YOUR CODE: return blocks to free list ---
        # Hint: self.free_blocks.extend(self.page_tables[request_id])
        pass  # YOUR CODE
        del self.page_tables[request_id]
        del self.seq_lengths[request_id]

    def memory_utilization(self):
        """Fraction of allocated memory actually used."""
        total_allocated = sum(len(blocks) * self.block_size
                             for blocks in self.page_tables.values())
        total_used = sum(self.seq_lengths.values())
        return total_used / total_allocated if total_allocated > 0 else 0

    def blocks_used(self):
        return self.total_blocks - len(self.free_blocks)


# --- YOUR CODE: demonstrate PagedAttention ---
block_size = 16
total_blocks = 1000
cache = SimplePagedKVCache(block_size, total_blocks)

# Add requests with varying lengths
np.random.seed(42)
request_lengths = np.random.exponential(scale=512, size=10).astype(int)
request_lengths = np.clip(request_lengths, 50, 4096)

for i, length in enumerate(request_lengths):
    cache.allocate_request(f"req_{i}", initial_tokens=length)

print(f"Block size: {block_size} tokens")
print(f"Total physical blocks: {total_blocks}")
print(f"Blocks in use: {cache.blocks_used()}")
print(f"Memory utilization: {cache.memory_utilization():.1%}")
print(f"\nCompare to static allocation:")
# Static: max_len / block_size blocks per request
static_blocks = len(request_lengths) * (4096 // block_size)
print(f"  Static would need: {static_blocks} blocks")
print(f"  PagedAttention uses: {cache.blocks_used()} blocks")
print(f"  Savings: {(1 - cache.blocks_used() / static_blocks) * 100:.1f}%")

# Show page table for one request
print(f"\nPage table for req_0 (length={request_lengths[0]}):")
print(f"  Physical blocks: {cache.page_tables['req_0'][:10]}...")
print(f"  Blocks allocated: {len(cache.page_tables['req_0'])}")
print(f"  Internal fragmentation: {len(cache.page_tables['req_0']) * block_size - request_lengths[0]} tokens wasted")

## 5. How vLLM Works

**vLLM** is the reference implementation of PagedAttention + continuous batching.

Architecture:
1. **Scheduler**: decides which requests to run each step (continuous batching)
2. **Block Manager**: manages physical KV cache blocks (PagedAttention)
3. **Model Runner**: executes the model on the selected batch
4. **PagedAttention kernel**: custom CUDA kernel that reads KV cache from scattered blocks

Key features:
- (a) **PagedAttention** for near-zero KV cache waste
- (b) **Continuous batching** for high throughput
- (c) **Preemption** (swap/recompute) for fairness when memory is tight

Let's install vLLM and serve LLaMA-1B.

In [ ]:
# Install vLLM (uncomment to run)
# !pip install vllm

# --- YOUR CODE: serve LLaMA-1B with vLLM and measure throughput ---
# Hint:
# from vllm import LLM, SamplingParams
# llm = LLM(model="meta-llama/Llama-3.2-1B", dtype="float16")
# sampling_params = SamplingParams(temperature=0.8, max_tokens=128)

# Benchmark at different concurrency levels
# import time
# concurrency_levels = [1, 4, 16, 64]
# results = []
#
# for n in concurrency_levels:
#     prompts = ["The future of AI is"] * n
#     start = time.time()
#     outputs = llm.generate(prompts, sampling_params)
#     elapsed = time.time() - start
#     total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
#     throughput = total_tokens / elapsed
#     results.append({'concurrency': n, 'throughput': throughput,
#                     'latency_ms': elapsed / total_tokens * 1000})
#     print(f"Concurrency {n:>3}: {throughput:.0f} tok/s, {elapsed/total_tokens*1000:.1f} ms/tok")

# Simulated results (replace with real measurements on your GPU)
print("Expected results on A100 (LLaMA-3.2-1B FP16):")
print(f"  Concurrency  1: ~150 tok/s,  ~6.7 ms/tok")
print(f"  Concurrency  4: ~550 tok/s,  ~7.3 ms/tok")
print(f"  Concurrency 16: ~1800 tok/s, ~8.9 ms/tok")
print(f"  Concurrency 64: ~4500 tok/s, ~14.2 ms/tok")
print(f"\nThroughput scales ~30x from batch=1 to batch=64!")
print(f"But per-request latency increases ~2x — the throughput-latency tradeoff.")

## 6. Throughput vs Latency Tradeoff

Bigger batches = higher **throughput** (total tokens/sec across all users) but higher **per-request latency** (ms/token for each user).

Why? With more requests:
- Weight loading is amortized -> throughput up
- But each forward pass takes longer (more tokens to process) -> latency up
- Also: larger KV caches contend for memory bandwidth

The **sweet spot** depends on your SLA: what's the maximum acceptable per-token latency?

In [ ]:
# --- YOUR CODE: benchmark vLLM at different batch sizes ---
# If you have vLLM installed, replace simulated data with real measurements.

# Simulated data (LLaMA-3.2-1B FP16 on A100)
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]

# --- YOUR CODE: fill in measured or estimated values ---
# Hint: throughput grows roughly linearly until memory-bound,
# latency grows sub-linearly, then sharply at high batch
throughputs = [150, 290, 550, 1000, 1800, 3000, 4500, 5500]  # YOUR CODE: tok/s
latencies = [6.7, 6.9, 7.3, 8.0, 8.9, 10.7, 14.2, 23.3]    # YOUR CODE: ms/tok

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Throughput plot
ax1.plot(batch_sizes, throughputs, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Batch Size (concurrent requests)', fontsize=12)
ax1.set_ylabel('Throughput (tokens/sec)', fontsize=12)
ax1.set_title('Throughput vs Batch Size', fontsize=14, fontweight='bold')
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=peak_tflops * 1e12 / (total_params * 2 * 2) , color='r', linestyle='--',
            alpha=0.5, label='Theoretical max (compute-bound)')
ax1.legend()

# Latency plot
ax2.plot(batch_sizes, latencies, 'r-s', linewidth=2, markersize=8)
ax2.set_xlabel('Batch Size (concurrent requests)', fontsize=12)
ax2.set_ylabel('Latency (ms/token)', fontsize=12)
ax2.set_title('Per-Request Latency vs Batch Size', fontsize=14, fontweight='bold')
ax2.set_xscale('log', base=2)
ax2.grid(True, alpha=0.3)

# Mark sweet spot
sweet_spot_idx = 4  # batch=16
ax1.annotate('Sweet spot?', xy=(batch_sizes[sweet_spot_idx], throughputs[sweet_spot_idx]),
             xytext=(batch_sizes[sweet_spot_idx]*2, throughputs[sweet_spot_idx]*0.7),
             arrowprops=dict(arrowstyle='->', color='green'), fontsize=11, color='green')
ax2.annotate('Sweet spot?', xy=(batch_sizes[sweet_spot_idx], latencies[sweet_spot_idx]),
             xytext=(batch_sizes[sweet_spot_idx]*2, latencies[sweet_spot_idx]*1.3),
             arrowprops=dict(arrowstyle='->', color='green'), fontsize=11, color='green')

plt.tight_layout()
plt.show()

print(f"\nSweet spot analysis (batch=16):")
print(f"  Throughput: {throughputs[sweet_spot_idx]} tok/s ({throughputs[sweet_spot_idx]/throughputs[0]:.0f}x vs batch=1)")
print(f"  Latency: {latencies[sweet_spot_idx]} ms/tok ({latencies[sweet_spot_idx]/latencies[0]:.1f}x vs batch=1)")
print(f"  Good trade: {throughputs[sweet_spot_idx]/throughputs[0]:.0f}x throughput for only {latencies[sweet_spot_idx]/latencies[0]:.1f}x latency")

## 7. Memory Efficiency Comparison

Compare memory usage for 32 concurrent requests on LLaMA-3.2-1B with varying sequence lengths:

- **(a) Naive pre-allocation**: reserve `max_seq_len` for every request
- **(b) PagedAttention**: allocate blocks on demand, only for tokens actually generated

This directly determines how many concurrent requests your GPU can handle.

In [ ]:
n_concurrent = 32
max_seq_len = 4096
block_size = 16

# Generate realistic varying sequence lengths
np.random.seed(123)
actual_lengths = np.random.exponential(scale=400, size=n_concurrent).astype(int)
actual_lengths = np.clip(actual_lengths, 32, max_seq_len)

# --- YOUR CODE: calculate memory for both approaches ---
# Hint: kv_per_token already calculated above

# (a) Naive pre-allocation
naive_mem = None   # YOUR CODE: n_concurrent * max_seq_len * kv_per_token

# (b) PagedAttention: round each request up to block boundary
paged_blocks_per_req = np.ceil(actual_lengths / block_size).astype(int)
paged_mem = None   # YOUR CODE: sum(paged_blocks_per_req) * block_size * kv_per_token

# Actual minimum (no fragmentation at all)
ideal_mem = None   # YOUR CODE: sum(actual_lengths) * kv_per_token

savings = None     # YOUR CODE: (1 - paged_mem / naive_mem) * 100

print(f"32 concurrent requests, max_seq_len={max_seq_len}")
print(f"Actual lengths: mean={actual_lengths.mean():.0f}, max={actual_lengths.max()}, min={actual_lengths.min()}")
print(f"\n{'Method':<25} {'Memory (GB)':>12} {'vs Naive':>10}")
print("="*50)
print(f"{'Naive pre-allocation':<25} {naive_mem/1e9:>12.2f} {'100%':>10}")
print(f"{'PagedAttention':<25} {paged_mem/1e9:>12.2f} {paged_mem/naive_mem*100:>9.1f}%")
print(f"{'Ideal (no waste)':<25} {ideal_mem/1e9:>12.2f} {ideal_mem/naive_mem*100:>9.1f}%")
print(f"\nPagedAttention saves {savings:.1f}% of KV cache memory")
print(f"Internal fragmentation (paged vs ideal): {(paged_mem - ideal_mem)/ideal_mem*100:.1f}%")

# How many more requests can we serve?
gpu_mem = 80e9  # A100 80GB
model_mem = weight_bytes
available = gpu_mem - model_mem
max_naive = int(available / (max_seq_len * kv_per_token))
max_paged = int(available / (actual_lengths.mean() * kv_per_token * (1 + block_size/actual_lengths.mean())))
print(f"\nA100 80GB (after model weights):")
print(f"  Naive: {max_naive} concurrent requests max")
print(f"  Paged: ~{max_paged} concurrent requests max")

## 8. Prefix Caching and Prompt Sharing

When multiple requests share the **same system prompt** (common in production), their KV cache
for those shared tokens is **identical**.

PagedAttention enables **copy-on-write** sharing:
- Shared prefix tokens map to the **same physical blocks**
- Only unique suffix tokens get their own blocks
- Like copy-on-write in OS virtual memory: share until someone writes

This is especially powerful for:
- Chat applications with long system prompts
- RAG with shared document context
- Multi-turn conversations sharing history

In [ ]:
# Prefix caching: 100 requests sharing a 2048-token system prompt
n_requests = 100
shared_prefix_len = 2048  # tokens
unique_suffix_len = 512   # average unique tokens per request

# --- YOUR CODE: calculate memory with and without sharing ---

# Without sharing: each request stores full KV cache (prefix + suffix)
no_share_mem = None  # YOUR CODE: n_requests * (shared_prefix_len + unique_suffix_len) * kv_per_token

# With sharing: ONE copy of prefix + per-request suffix
shared_mem = None    # YOUR CODE: (shared_prefix_len + n_requests * unique_suffix_len) * kv_per_token

savings_gb = None    # YOUR CODE: (no_share_mem - shared_mem) / 1e9
savings_pct = None   # YOUR CODE: (1 - shared_mem / no_share_mem) * 100

print(f"Prefix Caching Analysis")
print(f"  {n_requests} requests sharing {shared_prefix_len}-token system prompt")
print(f"  Each request generates {unique_suffix_len} unique tokens")
print(f"\n{'Without sharing':<20}: {no_share_mem/1e9:.2f} GB")
print(f"{'With sharing':<20}: {shared_mem/1e9:.2f} GB")
print(f"{'Savings':<20}: {savings_gb:.2f} GB ({savings_pct:.1f}%)")
print(f"\nShared prefix is {shared_prefix_len/(shared_prefix_len+unique_suffix_len)*100:.0f}% of total tokens")
print(f"Memory saving is {savings_pct:.0f}% — almost directly proportional!")

# Compute savings too: prefix KV only computed once
# Prefill FLOPs for prefix
prefix_flops_per_layer = 2 * shared_prefix_len * d_model * (n_kv_heads * d_head) * 2  # K + V projections
prefix_flops_total = prefix_flops_per_layer * n_layers
saved_compute = prefix_flops_total * (n_requests - 1)  # Only compute once instead of n times
print(f"\nCompute savings from shared prefix:")
print(f"  {saved_compute / 1e12:.2f} TFLOPS saved ({(n_requests-1)/n_requests*100:.0f}% of prefix compute)")

## 9. Production Serving Checklist

A production LLM serving stack combines multiple optimizations:

| Optimization | Chapter | What it does | Speedup |
|---|---|---|---|
| Continuous batching | Ch9 | Maximize GPU utilization across requests | 10-30x throughput |
| PagedAttention | Ch9 | Eliminate KV cache memory waste | 2-5x more concurrent requests |
| Quantized weights | Ch7 | INT8/INT4 weights -> less memory bandwidth | 2-4x decode speed |
| Quantized KV cache | Ch7 | FP8/INT8 KV cache -> more requests fit | 1.5-2x more concurrent |
| Flash Attention | Ch4 | Fused attention kernel -> less HBM traffic | 2-4x attention speedup |
| Kernel fusion / torch.compile | Ch6 | Fuse elementwise ops | 1.2-1.5x overall |

Estimate the total improvement by combining everything.

In [ ]:
# --- YOUR CODE: estimate total throughput improvement ---
# Each optimization has an approximate speedup range.
# Some are multiplicative, some overlap.

# Baseline: batch=1 naive decode on LLaMA-3.2-1B FP16
# ~150 tok/s on A100
baseline_tps = 150  # tokens/sec

# Hint: think about which speedups stack multiplicatively
# Continuous batching: ~12x throughput (batch=1 -> batch=16 effective)
batch_speedup = None        # YOUR CODE: 12
# Quantization (W8A16): ~1.8x (halve weight bytes -> nearly 2x decode speed)
quant_speedup = None        # YOUR CODE: 1.8
# Flash Attention: ~1.3x overall (attention is ~20-30% of total, Flash is 2-4x on that)
flash_speedup = None        # YOUR CODE: 1.3
# Kernel fusion: ~1.2x (elementwise is ~5-10% of total)
fusion_speedup = None       # YOUR CODE: 1.2
# PagedAttention: enables 3x more concurrent requests -> more batching headroom
paged_multiplier = None     # YOUR CODE: 1.5 (indirect via enabling larger batches)

total_speedup = batch_speedup * quant_speedup * flash_speedup * fusion_speedup * paged_multiplier
optimized_tps = baseline_tps * total_speedup

print(f"Production Throughput Estimate (LLaMA-3.2-1B on A100)")
print(f"={'='*55}")
print(f"{'Optimization':<30} {'Speedup':>10} {'Cumulative tok/s':>18}")
print(f"{'-'*55}")

cumulative = baseline_tps
print(f"{'Baseline (batch=1 FP16)':<30} {'1.0x':>10} {cumulative:>15.0f}")

for name, factor in [('+ Continuous batching', batch_speedup),
                     ('+ Weight quantization', quant_speedup),
                     ('+ Flash Attention', flash_speedup),
                     ('+ Kernel fusion', fusion_speedup),
                     ('+ PagedAttention headroom', paged_multiplier)]:
    cumulative *= factor
    print(f"{name:<30} {f'{factor:.1f}x':>10} {cumulative:>15.0f}")

print(f"{'-'*55}")
print(f"{'TOTAL':<30} {f'{total_speedup:.0f}x':>10} {optimized_tps:>15.0f}")
print(f"\nFrom {baseline_tps} tok/s to {optimized_tps:.0f} tok/s = {total_speedup:.0f}x improvement")
print(f"\nNote: These are rough estimates. Real gains depend on workload,")
print(f"sequence lengths, and how well optimizations compose.")

## Revision Notes

*Fill this in after running all sections.*

---

**Static vs continuous batching:**
- Static utilization: ___%
- Continuous utilization: ___%
- Why continuous is better: ___

**KV cache memory per token (LLaMA-3.2-1B FP16):**
- Per token per layer: ___ bytes
- Per token all layers: ___ KB

**PagedAttention key ideas:**
- Block size: ___ tokens
- Memory savings vs pre-allocation: ___% 
- Inspired by: ___

**Throughput vs latency sweet spot:**
- Best batch size for my workload: ___
- Throughput at that batch: ___ tok/s
- Latency at that batch: ___ ms/tok

**Prefix caching savings:**
- Memory saved with shared prompt: ___%

**Total production stack speedup:** ___x

**What surprised me:**



---
*Next: Chapter 10 — Speculative Decoding & Advanced Inference*